# Healthcare Readmission Prediction
## 01 — Data Cleaning and Preprocessing

This notebook performs the first data-analysis stage for the Diabetes 130-US Hospitals dataset.

## Objectives
- Load the public UCI dataset
- Inspect rows, columns and data types
- Convert `?` to missing values
- Calculate missing-value counts and percentages
- Check duplicate records
- Generate numerical summaries
- Calculate the 30-day readmission rate
- Create a binary modelling target

In [14]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 100)


## Load the dataset
The dataset can be obtained from the UCI Machine Learning Repository using `ucimlrepo`.

In [15]:
# Run this line once if the package is not installed:
%pip install ucimlrepo

from ucimlrepo import fetch_ucirepo

diabetes = fetch_ucirepo(id=296)
X = diabetes.data.features.copy()
y = diabetes.data.targets.copy()
df = pd.concat([X, y], axis=1)

print('Dataset loaded successfully')
print('Rows:', df.shape[0])
print('Columns:', df.shape[1])

/usr/local/lib/python3.13/dist-packages/ucimlrepo/fetch.py:97: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_url)


Dataset loaded successfully
Rows: 101766
Columns: 48


## 1. Initial assessment

In [16]:
print('Shape:', df.shape)
print('\nFirst five records:')
display(df.head())
print('\nData types:')
display(df.dtypes.value_counts())
print('\nDataset information:')
df.info()

Shape: (101766, 48)

First five records:


,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,Caucasian,Female,[0-10),NaN,6,25,1,1,NaN,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,NaN,NaN,1,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO
1,Caucasian,Female,[10-20),NaN,1,1,7,3,NaN,NaN,59,0,18,0,0,0,276,250.01,255,9,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30
2,AfricanAmerican,Female,[20-30),NaN,1,1,7,2,NaN,NaN,11,5,13,2,0,1,648,250,V27,6,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO
3,Caucasian,Male,[30-40),NaN,1,1,7,2,NaN,NaN,44,1,16,0,0,0,8,250.43,403,7,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO
4,Caucasian,Male,[40-50),NaN,1,1,7,1,NaN,NaN,51,0,8,0,0,0,197,157,250,5,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO



Data types:


,count
object,37
int64,11



Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 48 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   race                      99493 non-null   object
 1   gender                    101766 non-null  object
 2   age                       101766 non-null  object
 3   weight                    3197 non-null    object
 4   admission_type_id         101766 non-null  int64 
 5   discharge_disposition_id  101766 non-null  int64 
 6   admission_source_id       101766 non-null  int64 
 7   time_in_hospital          101766 non-null  int64 
 8   payer_code                61510 non-null   object
 9   medical_specialty         51817 non-null   object
 10  num_lab_procedures        101766 non-null  int64 
 11  num_procedures            101766 non-null  int64 
 12  num_medications           101766 non-null  int64 
 13  number_outpatient         101766 non-

## 2. Convert `?` to missing values

In [17]:
question_marks = (df == '?').sum().sum()
print('Total ? placeholders before conversion:', int(question_marks))

df = df.replace('?', np.nan)
print('Total missing values after conversion:', int(df.isna().sum().sum()))

Total ? placeholders before conversion: 0
Total missing values after conversion: 374017


## 3. Missing-value summary

Missing percentage = missing observations / total observations × 100

In [18]:
missing_summary = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_percent': (df.isna().mean() * 100).round(2)
}).sort_values('missing_percent', ascending=False)

display(missing_summary.head(15))

,missing_count,missing_percent
weight,98569,96.86
max_glu_serum,96420,94.75
A1Cresult,84748,83.28
medical_specialty,49949,49.08
payer_code,40256,39.56
race,2273,2.23
diag_3,1423,1.40
diag_2,358,0.35
diag_1,21,0.02
time_in_hospital,0,0.00


## 4. Duplicate records

In [19]:
duplicate_count = int(df.duplicated().sum())
duplicate_percent = duplicate_count / len(df) * 100

print('Duplicate rows:', duplicate_count)
print(f'Duplicate percentage: {duplicate_percent:.2f}%')

Duplicate rows: 0
Duplicate percentage: 0.00%


## 5. Numerical descriptive statistics

In [20]:
numeric_columns = df.select_dtypes(include=np.number).columns
numeric_summary = df[numeric_columns].describe().T
display(numeric_summary)

,count,mean,std,min,25%,50%,75%,max
admission_type_id,101766.0,2.024006,1.445403,1.0,1.0,1.0,3.0,8.0
discharge_disposition_id,101766.0,3.715642,5.280166,1.0,1.0,1.0,4.0,28.0
admission_source_id,101766.0,5.754437,4.064081,1.0,1.0,7.0,7.0,25.0
time_in_hospital,101766.0,4.395987,2.985108,1.0,2.0,4.0,6.0,14.0
num_lab_procedures,101766.0,43.095641,19.674362,1.0,31.0,44.0,57.0,132.0
num_procedures,101766.0,1.339730,1.705807,0.0,0.0,1.0,2.0,6.0
num_medications,101766.0,16.021844,8.127566,1.0,10.0,15.0,20.0,81.0
number_outpatient,101766.0,0.369357,1.267265,0.0,0.0,0.0,0.0,42.0
number_emergency,101766.0,0.197836,0.930472,0.0,0.0,0.0,0.0,76.0
number_inpatient,101766.0,0.635566,1.262863,0.0,0.0,0.0,1.0,21.0


## 6. Readmission distribution and 30-day rate

In [21]:
target_counts = df['readmitted'].value_counts(dropna=False)
display(target_counts)

early_readmissions = int(target_counts.get('<30', 0))
total_encounters = len(df)
rate = early_readmissions / total_encounters * 100

print(f'Early readmissions (<30 days): {early_readmissions:,}')
print(f'Total encounters: {total_encounters:,}')
print(f'30-day readmission rate: {rate:.2f}%')

,count
readmitted,
NO,54864
>30,35545
<30,11357


Early readmissions (<30 days): 11,357
Total encounters: 101,766
30-day readmission rate: 11.16%


## 7. Create a binary modelling target

`1` represents readmission within 30 days (`<30`). `0` represents `>30` or `NO`. The original `readmitted` column is retained.

In [22]:
df['readmitted_30d'] = (df['readmitted'] == '<30').astype(int)

print('Binary target counts:')
display(df['readmitted_30d'].value_counts())

print('Binary target percentages:')
display((df['readmitted_30d'].value_counts(normalize=True) * 100).round(2))

Binary target counts:


,count
readmitted_30d,
0,90409
1,11357


Binary target percentages:


,proportion
readmitted_30d,
0,88.84
1,11.16


## 8. Data-cleaning decision log

| Issue | Method | Reason |
|---|---|---|
| `?` placeholders | Convert to `NaN` | Treat unavailable information consistently |
| Missing values | Count and percentage | Identify variables needing review |
| Duplicates | Programmatic check | Prevent accidental duplicate observations |
| Numerical variables | Descriptive statistics | Understand range and distribution |
| Readmission target | Preserve original + create binary target | Support later classification modelling |

**Quality note:** Very high missingness should not automatically be replaced with a mean or mode. Each variable should be reviewed and the final treatment documented before modelling.

## 9. Final check

In [23]:
print('Final rows:', len(df))
print('Final columns:', len(df.columns))
print('Total missing cells:', int(df.isna().sum().sum()))
print('Notebook data-cleaning assessment completed.')

Final rows: 101766
Final columns: 49
Total missing cells: 374017
Notebook data-cleaning assessment completed.


In [24]:
target_counts = df['readmitted'].value_counts(dropna=False)
display(target_counts)

early_readmissions = int(target_counts.get('<30', 0))
total_encounters = len(df)
rate = early_readmissions / total_encounters * 100

print(f'Early readmissions (<30 days): {early_readmissions:,}')
print(f'Total encounters: {total_encounters:,}')
print(f'30-day readmission rate: {rate:.2f}%')

,count
readmitted,
NO,54864
>30,35545
<30,11357


Early readmissions (<30 days): 11,357
Total encounters: 101,766
30-day readmission rate: 11.16%


In [25]:
df['readmitted_30d'] = (df['readmitted'] == '<30').astype(int)

print('Binary target counts:')
display(df['readmitted_30d'].value_counts())

print('Binary target percentages:')
display((df['readmitted_30d'].value_counts(normalize=True) * 100).round(2))

Binary target counts:


,count
readmitted_30d,
0,90409
1,11357


Binary target percentages:


,proportion
readmitted_30d,
0,88.84
1,11.16
